# OneAI Agent 框架 — 技术面试问题列表

> 基于简历内容，按面试深度分层组织。面试官视角：验证简历真实深度 + 探测边界认知。

---

## 第一层：项目全局与设计决策（验证架构师视角）

### 1. 项目定位与竞品认知

1. **你做竞品分析时对比了 15+ 竞品、24 个维度，最终 OneAI 的定位和差异化在哪里？** 如果今天让你重新定位，会调整什么？
2. **你说"不依赖任何现有 Agent SDK"，这个决策的 trade-off 是什么？** 举例：LangChain 的 tool calling 抽象已经够用了，为什么要从 LlmProvider trait 开始自己写？
3. **Claude Code / Codex / OpenCode 这三者各自的架构核心是什么？** 你在竞品分析中认为 OneAI 最关键的 3 个差距是什么？现在补齐了几个？

### 2. 整体架构

4. **19 个 crate 的拆分逻辑是什么？** 为什么 memory 和 rag 是分开的？为什么 domain 和 skill 是分开的？有没有出现过"应该合并"或"应该拆得更细"的情况？
5. **AppBuilder 是所有模块的组装入口，它和 Spring 的 DI 容器有什么本质区别？** 如果用户不需要 RAG 但需要工具，AppBuilder 如何保证"按需使用"？
6. **你提到跨平台支持六端，UniFFI 的绑定在实际项目里跑通了哪些？** Kotlin/Swift/C++ 的 binding 生成有没有遇到类型映射的坑？async trait 在 UniFFI 里怎么处理的？

---

## 第二层：核心模块深挖（验证实现深度）

### DomainPack 系统

7. **DomainPack 5层的拆分依据是什么？** 为什么"工具描述覆写"（ToolDecorator）是第1层而不是最上层？如果只保留 3 层，你会砍掉哪两层？
8. **CodingPack 的 4 个范式策略（重构→Plan+ReAct+Reflect，修bug→Plan+ReAct…）是怎么得出这个映射的？** 是经验总结还是有数据支撑？如果用户任务既不是重构也不是修bug，范式策略如何 fallback？
9. **DomainPack 合并的"严格优先"规则具体是什么？** coding + research 合并时，如果 coding 说 Shell 要 require_confirmation 但 research 说 auto_approve，谁赢？这个规则有边界情况吗？
10. **PackSource + PackRegistry + builtin index 的设计是什么？** DomainPack 从"声明式配置"到"市场级分发"跨越了什么问题？注册一个第三方 Pack 的流程是怎样的？

### Agentic Loop

11. **动态决策的 4 种类型（DirectAnswer/ToolCalls/Delegate/SwitchParadigm）是模型输出一个 enum 还是通过某种启发式判断的？** 如果模型输出不在这 4 种里怎么处理？
12. **TokenBudget 约束替代 max_iterations，具体怎么实现的？** 预用完了但 Agent 还没得出结论时，是强制终止还是触发压缩后继续？token 计数用的是 LLM 返回的 usage 还是本地估算？
13. **你说"非固定管线"，但 AgentLoop 的代码里 `run_iteration()` 是顺序调用的——"动态"在哪里体现？** 是每轮重新选择 paradigm 还是每轮 model 自由输出决定下一步？

### 子代理系统

14. **SubAgent 在独立 worktree 中执行，worktree 的创建/销毁/冲突处理谁负责？** 如果两个 SubAgent 同时修改同一个文件怎么办？
15. **SubAgent 的工具子集过滤是怎么做的？** searcher 只有 read+grep+glob——这个约束是在 DomainPack 的 SubAgentTypeDefinition 里声明还是在 AgentLoop 启动时动态裁剪？如果 SubAgent 在执行中发现自己需要额外工具怎么办？
16. **子代理返回的 SubAgentSummary 是结构化数据还是自由文本？** 竞品分析里你标注过"SubAgentSummary 是自由文本"是差距——现在改了吗？

### StateGraph 与 Workflow

17. **StateGraph 和 WorkflowDag 的本质区别是什么？** DAG 无环，StateGraph 有环——为什么需要两种？ReAct 循环用 StateGraph 表达，那 Plan 的有序步骤用 DAG 还是 StateGraph？
18. **StateGraph 编译到执行器（Compile → Execute）的过程是什么？** 编译阶段做了什么校验？如果有环但缺少退出条件，编译器能检测出来吗？
19. **StateGraph 和 AgentLoop 的闭环集成是怎么接的？** 具体是 AgentLoop 在某轮迭代发现需要进入 StateGraph，还是 StateGraph 的节点本身就是 AgentLoop？

---

## 第三层：LLM 工程关键技术（验证对大模型的理解深度）

### 输出解析

20. **3层解析防御的约束解码层，BNF 语法是怎么引导模型输出的？** 是用 grammar-constrained decoding（如 Llama.cpp 的 GBNF）还是仅作为 prompt 中的格式指令？如果是后者，模型真的会遵守 BNF 吗？
21. **模糊 JSON 修复的"嵌入式 JSON 检测"是什么意思？** 给一个实际例子：模型输出什么样的脏数据，第2层能修但第1层不行？
22. **回退自纠（重新提示模型修正）的 prompt 是怎么构造的？** 把原始输出连同错误信息喂回去，模型通常能修正吗？你测过成功率吗？有没有出现"越纠越错"的情况？

### 流式处理

23. **StreamParser 实时检测 tool_use 块，Anthropic 的 streaming API 返回的是 `input_json_delta` 片段——你怎么在参数还没生成完的时候就判断"这是一个 tool call"？** 具体的状态机逻辑是什么？
24. **增量解析和完整解析的性能差异有多大？** 如果网络抖动导致 chunk 丢失或乱序，StreamParser 怎么恢复？

### 上下文管理

25. **ContextBudgetManager "按比例分配上下文预算"，比例是怎么定的？** system prompt 占多少、工具描述占多少、对话历史占多少——是静态比例还是动态计算？
26. **每轮自动压缩触发时，压缩的策略是什么？** 是摘要压缩还是截断保留最近 N 条？摘要压缩用的什么模型？如果压缩本身消耗了预算怎么办？
27. **EnvironmentDiff 检测的"环境差异"具体检测什么？** git status 变化、文件树变化——这些信息是每轮都 diff 还是只在特定事件触发时 diff？性能开销怎么控制？

---

## 第四层：工具、安全与协议（验证工程落地能力）

### MCP 协议

28. **你用 rmcp crate 实现 MCP，rmcp 本身提供了什么能力？** 你在它之上做了哪些扩展？MCP 的 stdio 传输的子进程生命周期管理怎么做的（竞品分析里你标注过这个差距）？
29. **MCP handshake 的实现状态是什么？** 你标注过"SSE/StreamableHttp 未实现"——现在补齐了吗？如果没补齐，在生产场景下 MCP server 大多用 SSE 传输，OneAI 怎么对接？

### WASM 沙箱

30. **WASM 沙箱执行引擎和 ShellTool 的 regex 黑名单是替代关系还是并行关系？** 如果一个工具既能在 WASM 里跑又有 Shell 版本，怎么选？
31. **WasmTool 的输入输出是怎么传递的？** WASM 模块本身不能访问文件系统——工具需要读文件时，是通过宿主函数（host function）注入的还是预先把文件内容传进去？
32. **Wasmtime 的编译延迟和内存开销在实际场景下有多大？** 257 个测试里有 WASM 相关的性能测试吗？

### 权限与审批门

33. **PermissionLevel 的 Read/Standard/Full 三级是怎么映射到具体工具的？** GrepTool 是 Read 还是 Standard？MCP 调用是 Standard 还是 Full？一个工具的 risk_level 和 permission_level 如果冲突了怎么办？
34. **ChannelApprovalGate 和 PlatformApprovalGate 的区别是什么？** Channel 是异步的（发到 UI 等人回），Platform 是同步弹对话框——在实际跨平台场景中，Android 上用哪个？
35. **DomainPack 覆写权限的 5 级优先级链（deny_by_default → permission_overrides → auto_approve → require_confirmation → fallback risk_level），为什么 deny_by_default 是最高优先级而不是最低？** 这个设计的安全哲学是什么？

---

## 第五层：记忆、RAG 与评测（验证系统性思维）

### 记忆系统

36. **STM 的滑动窗口驱逐到 LTM 时，驱逐的触发条件是什么？** 窗口满了驱逐最老的？还是按重要性排序驱逐？重要性怎么衡量？
37. **HNSW 向量存储是你自己实现的还是用第三方库？** 竞品分析提到"embedded lightweight implementation, no external dependency"——自实现的 HNSW 在召回率和性能上和 hnswlib 比过吗？
38. **长期记忆的"混合评分"是什么？** embedding 相似度 + 时间衰减 + 频次权重？各自的比例怎么定的？

### RAG

39. **EmbeddingService 支持 FastEmbed（本地 ONNX）/ Ollama / OpenAI，三者各自的适用场景是什么？** 生产环境下推荐哪个？FastEmbed 的 ONNX 模型在移动端跑过吗？
40. **分块策略的 SentenceBoundary 和 FixedSize 在实际效果上差异大吗？** 你测过不同策略对检索召回率的影响吗？

### 轨迹评测

41. **OpenInference 轨迹日志记录了"成功率"，但 Agent 的成功定义是什么？** 工具调用成功 ≠ 任务完成——你是怎么定义 Agent 执行的"成功"的？
42. **轨迹日志怎么和 OpenTelemetry 的 trace/span 模型对应？** 一个 AgentLoop 迭代是一个 span 还是一组 span？子代理的 span 怎么嵌套？

### Checkpoint

43. **渐进式 Checkpoint 的"渐进"是什么意思？** 是只存增量而非全量？还是逐步从内存→SQLite→Postgres 升级存储后端？
44. **SQLite 和 Postgres 的 Checkpoint 后端实际实现了吗？** 竞品分析里标注过 "SqliteCheckpoint 未实现"和"ProgressiveCheckpoint list todo!()"——现在状态如何？
45. **从任意检查点 fork 是什么意思？** fork 出的新分支和原来的会话是共享 LTM 还是各自独立？

---

## 第六层：Rust 工程能力（验证语言深度）

46. **sealed enum 层级的设计目的是什么？** 和普通的 enum + match 比，sealed trait 给你带来了什么好处？在哪个模块里用了这个模式？
47. **LlmProvider trait 用了 `#[async_trait]`，Rust 原生的 async trait 已经稳定了（1.75+），你为什么还用 async-trait crate？** 是历史原因还是有技术考量？
48. **tokio 的 `spawn` 在子代理执行里怎么用的？** 你标注过"未用 tokio::spawn 独立线程"——现在改了吗？如果多个 SubAgent 并行 spawn，它们的 ScopeState 隔离是怎么保证的？
49. **19 crate workspace 的编译时间有多长？** 有没有做增量编译优化？cargo 的 feature flag 有用来做可选依赖裁剪吗？
50. **UniFFI 绑定生成时，Rust 的 Result<T, E> 怎么映射到 Kotlin/Swift 的异常体系？** 自定义错误类型在跨语言时有什么坑？

---

## 第七层：跨界追问（验证思维迁移能力）

### 从移动端到 Agent

51. **你在畅连做音视频时，"随机丢包场景通过冗余度+多路传输提升质量"——这个思路和 Agent 的"LLM 输出不可靠通过 3 层解析防御修复"有什么思维共性？**
52. **你做安全 SE 时的 STRIDE 威胁分析，和 Agent 中 PermissionLevel + 审批门的设计有没有方法论上的延续？** 举例：STRIDE 的 Spoofing 怎么映射到 Agent 的安全威胁？
53. **Android Framework 的 AMS/WMS/PMS 调优，本质上是在有限资源下做调度优先级——这和 ContextBudgetManager 的"按比例分配上下文预算"是同一类问题吗？**

### 开放设计题

54. **如果让你给 OneAI 加一个 Multi-Agent 协作场景（3 个 Agent 协同完成一个复杂任务），你会怎么设计通信协议？** 用 A2A 还是自定义？共享记忆还是各自独立？冲突怎么解决？
55. **OneAI 目前的 DomainPack 是静态配置——如果要支持"Agent 在运行中发现新领域并动态加载 Pack"，架构要怎么改？**
56. **如果你加入一个 Agent 团队，第一件事你会做什么？** 是继续打磨 OneAI 还是用团队已有的框架？为什么？

---

## 面试官备忘：红线问题

> 以下问题如果答不上来，说明简历描述与实际深度不符：

| # | 红线问题 | 期望回答要点 |
|---|----------|-------------|
| 1 | DomainPack 5层各自的 `struct` 定义长什么样？ | 能说出至少3层的核心字段 |
| 2 | AgentLoop 一轮迭代的完整代码流（从接收用户输入到返回结果） | 能画出或口述至少 6 个步骤的调用链 |
| 3 | StreamParser 检测 tool_use 的状态机有几种状态？ | 能说出至少 3 种状态及其转换条件 |
| 4 | PermissionLevel 的 deny_by_default 为什么优先级最高？ | 能从安全设计哲学角度解释（fail-safe default） |
| 5 | 竞品分析中 OneAI 最关键的未解决差距是什么？ | 能说出至少 2 个 Critical 级差距的现状 |
